## EE8223 Deep Learning Project

## Extraction of Wav2Vec2 GENERIC Audio Embeddings from IEMOCAP Dataset for Validation Data

#Student: Jason Yip

### Description
This script extracts generic audio embeddings from the "validation" subset of the IEMOCAP dataset using the pretrained Wav2Vec2 model. The embeddings are generated through mean pooling of frame-level representations from the model's last hidden layer, providing a compact and meaningful representation for each audio file. The script associates each embedding with its corresponding emotion label and saves the results in a structured CSV file.

In [ ]:
import os
import zipfile
import pandas as pd
import torch
from transformers import Wav2Vec2Processor, Wav2Vec2Model
from tqdm import tqdm
import librosa
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Define path for the new zip file and extraction
zip_path = '/content/drive/MyDrive/Copy of IEMOCAP_new.zip'
extract_path = '/content/iemocap_data'

# Extract Copy of IEMOCAP_new.zip if not already extracted
if not os.path.exists(extract_path):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)

# Define paths to the CSV and the "validate" folder inside the extracted "IEMOCAP_Copied" folder
csv_path = os.path.join(extract_path, 'Copy of IEMOCAP', 'IEMOCAP_Copied', 'iemocap_full_dataset.csv')
validate_folder = os.path.join(extract_path, 'Copy of IEMOCAP', 'IEMOCAP_Copied', 'validate')

# Load the CSV file
data = pd.read_csv(csv_path)

# List all files in the validate folder and get their base filenames
validate_files = {filename.replace('.wav', '') for filename in os.listdir(validate_folder) if filename.endswith('.wav')}

# Filter the CSV to include only rows corresponding to files in the validate folder
validate_data = data[data['path'].apply(lambda x: x.replace('/', '_').replace('.wav', '') in validate_files)].reset_index(drop=True)

# Load the Wav2Vec 2.0 large model and processor from Hugging Face
processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-large-960h")
model = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-large-960h")

# Function to load audio, process it, and extract embeddings
def extract_embeddings(file_path):
    try:
        # Attempt to load the audio file with librosa at a sample rate of 16 kHz
        audio_input, _ = librosa.load(file_path, sr=16000)
    except Exception as e:
        # If loading fails, print a warning and skip the file
        print(f"Warning: Could not load {file_path}. Skipping. Error: {e}")
        return None, None  # Return None to indicate a problem with this file

    # Prepare the input values for the Wav2Vec 2.0 model
    input_values = processor(audio_input, sampling_rate=16000, return_tensors="pt").input_values

    # Extract embeddings using the model (without gradients as we're just using it for inference)
    with torch.no_grad():
        frame_embeddings = model(input_values).last_hidden_state

    # Mean pooling across frames to get a single embedding vector per audio file
    pooled_embedding = frame_embeddings.mean(dim=1).squeeze()

    # Return the embedding and its shape
    return pooled_embedding.numpy(), pooled_embedding.shape

# Initialize list to store embeddings
embeddings_list = []

# Process files in the filtered validate data order
for _, row in tqdm(validate_data.iterrows(), total=len(validate_data)):
    # Transform the path from the CSV to match the filename in the validate folder
    base_filename = row['path'].replace('/', '_').replace('.wav', '')
    file_path = os.path.join(validate_folder, base_filename + ".wav")  # Construct full path

    # Check if the file exists in the validate folder
    if not os.path.exists(file_path):
        continue  # Skip if the file is missing

    # Extract the emotion label
    emotion = row['emotion']

    # Extract embeddings from the WAV file
    embeddings, embedding_shape = extract_embeddings(file_path)

    # Skip adding to the list if embeddings were not extracted (e.g., file was corrupted)
    if embeddings is None:
        continue

    # Append embeddings, file path, label, and shape info to list
    embeddings_list.append({
        "file_path": base_filename,
        "folder": "validate",
        "emotion_label": emotion,
        "embedding_shape": embedding_shape,
        "embedding": embeddings
    })

# Convert list to DataFrame
embeddings_df = pd.DataFrame(embeddings_list)

# Convert embeddings to lists for saving in a CSV
embeddings_df['embedding'] = embeddings_df['embedding'].apply(lambda x: x.tolist())

# Save the DataFrame to CSV
embeddings_csv_path = "/content/iemocap_embeddings_validate.csv"
embeddings_df.to_csv(embeddings_csv_path, index=False)

# Display the DataFrame in Colab
print(embeddings_df)

print(f"Embeddings and metadata saved to {embeddings_csv_path}")

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/159 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/163 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/291 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/1.26G [00:00<?, ?B/s]

Some weights of Wav2Vec2Model were not initialized from the model checkpoint at facebook/wav2vec2-large-960h and are newly initialized: ['wav2vec2.encoder.pos_conv_embed.conv.parametrizations.weight.original0', 'wav2vec2.encoder.pos_conv_embed.conv.parametrizations.weight.original1', 'wav2vec2.masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
100%|██████████| 1505/1505 [12:15<00:00,  2.05it/s]


                                              file_path    folder  \
0     Session1_sentences_wav_Ses01F_script02_1_Ses01...  validate   
1     Session1_sentences_wav_Ses01F_script02_1_Ses01...  validate   
2     Session1_sentences_wav_Ses01F_script02_1_Ses01...  validate   
3     Session1_sentences_wav_Ses01F_script02_1_Ses01...  validate   
4     Session1_sentences_wav_Ses01F_script02_1_Ses01...  validate   
...                                                 ...       ...   
1500  Session5_sentences_wav_Ses05F_impro06_Ses05F_i...  validate   
1501  Session5_sentences_wav_Ses05F_impro06_Ses05F_i...  validate   
1502  Session5_sentences_wav_Ses05F_impro06_Ses05F_i...  validate   
1503  Session5_sentences_wav_Ses05F_impro06_Ses05F_i...  validate   
1504  Session5_sentences_wav_Ses05F_impro06_Ses05F_i...  validate   

     emotion_label embedding_shape  \
0              fru         (1024,)   
1              neu         (1024,)   
2              neu         (1024,)   
3              neu 